In [0]:
%sql
CREATE OR REPLACE TABLE delivery.gold.orders_per_5min AS
SELECT w.start AS window_start, w.end AS window_end,
       count(*) AS orders, round(sum(amount), 2) AS revenue
FROM (
  SELECT window(placed_at, '5 minutes') AS w, amount
  FROM delivery.silver.orders
  WHERE placed_at IS NOT NULL
)
GROUP BY w;

In [0]:
%sql
-- "Late" threshold is 30 seconds because the generator finishes orders in seconds.
-- For real delivery data you would use something like 45 minutes (2700 seconds).
CREATE OR REPLACE TABLE delivery.gold.delivery_metrics AS
SELECT
  area,
  count(*) AS total_orders,
  sum(CASE WHEN status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
  round(avg(CASE WHEN status = 'delivered' AND placed_at IS NOT NULL
        THEN unix_timestamp(delivered_at) - unix_timestamp(placed_at) END), 1) AS avg_delivery_seconds,
  round(
    sum(CASE WHEN status = 'delivered' AND placed_at IS NOT NULL
         AND unix_timestamp(delivered_at) - unix_timestamp(placed_at) > 30 THEN 1 ELSE 0 END)
    / nullif(sum(CASE WHEN status = 'delivered' AND placed_at IS NOT NULL THEN 1 ELSE 0 END), 0)
  , 3) AS late_rate,
  round(sum(amount), 2) AS revenue
FROM delivery.silver.orders
GROUP BY area;

In [0]:
%sql
SELECT
  (SELECT sum(orders) FROM delivery.gold.orders_per_5min)            AS gold_window_orders,
  (SELECT count(*) FROM delivery.silver.orders WHERE placed_at IS NOT NULL) AS silver_orders_with_placed,
  (SELECT sum(total_orders) FROM delivery.gold.delivery_metrics)     AS gold_area_orders,
  (SELECT count(*) FROM delivery.silver.orders)                      AS silver_orders_all,
  (SELECT round(sum(revenue), 2) FROM delivery.gold.delivery_metrics) AS gold_revenue,
  (SELECT round(sum(amount), 2) FROM delivery.silver.orders)         AS silver_revenue;

In [0]:
%sql
SELECT * FROM delivery.gold.orders_per_5min ORDER BY window_start;
SELECT * FROM delivery.gold.delivery_metrics ORDER BY total_orders DESC;